In [1]:
!pip install -q nemo_toolkit[asr] lhotse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/

In [3]:
!pip install -U jiwer pandas soundfile librosa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 23.1 MB/s eta 0:00:00
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
  Attempting uninstall: librosa
    Found existing installation: librosa 0.11.0
    Uninstalling librosa-0.11.0:
      Successfully uninstalled librosa-0.11.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.1

In [2]:
from nemo.collections.asr.models import EncDecMultiTaskModel

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

import ast
import gc
import hashlib
import os
import re

import numpy as np
import pandas as pd
import torch

from jiwer import wer, cer
from tqdm.auto import tqdm


# ============================================================
# BASE PATHS
# ============================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/anon/"
    "pidgindataset2026October/"
)

METADATA_PATH = BASE_DIR / "root" / "Metadata.csv"
SPLIT_MANIFEST_PATH = BASE_DIR / "shared_split_manifest.csv"

# Fine-tuned models
WHISPER_ROOT = BASE_DIR / "PidginMed-ASR/whisper/hyperparameter_search/trial_01/selected_model/"
CANARY_ROOT = BASE_DIR / "PidginCanary/canary_hyperparameter_tuning/selected_best_model.nemo"
MEDASR_ROOT = BASE_DIR / "medasr_hf_clean/selected_model/"

# Output directory deliberately matches the statistical notebook
RESULTS_DIR = BASE_DIR / "Model_Results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


print("Metadata:", METADATA_PATH)
print("Split:", SPLIT_MANIFEST_PATH)

print("Whisper:", WHISPER_ROOT)
print("Canary:", CANARY_ROOT)
print("MedASR:", MEDASR_ROOT)

print("Results:", RESULTS_DIR)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [6]:
# ============================================================
# NORMALIZATION
# ============================================================

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ============================================================
# MEDICAL TERM PARSING
# ============================================================

def parse_medical_terms(value):

    if isinstance(value, list):
        terms = value

    elif pd.isna(value):
        terms = []

    elif isinstance(value, str):

        value = value.strip()

        if value in {
            "",
            "[]",
            "[ ]",
            "nan",
            "None",
            "null"
        }:
            terms = []

        else:

            try:
                terms = ast.literal_eval(value)

                if not isinstance(terms, list):
                    terms = [terms]

            except Exception:

                terms = [
                    x.strip()
                    for x in value.split(",")
                    if x.strip()
                ]

    else:
        terms = []


    cleaned = []
    seen = set()

    for term in terms:

        term = normalize_text(term)

        if term and term not in seen:
            cleaned.append(term)
            seen.add(term)

    return cleaned


# ============================================================
# MEDICAL TERM MATCHING
# ============================================================

def phrase_present(text, phrase):

    text = f" {normalize_text(text)} "
    phrase = f" {normalize_text(phrase)} "

    return phrase in text


def reference_medical_terms(reference, terms):

    candidates = set(
        parse_medical_terms(terms)
    )

    return {
        term
        for term in candidates
        if phrase_present(reference, term)
    }


def predicted_medical_terms(prediction, terms):

    candidates = set(
        parse_medical_terms(terms)
    )

    return {
        term
        for term in candidates
        if phrase_present(prediction, term)
    }

In [7]:
# ============================================================
# ORIGINAL CORPUS-LEVEL mWER LOGIC
# ============================================================

def medical_wer(
    predictions,
    references,
    medical_terms_list
):

    ref_med_sentences = []
    pred_med_sentences = []

    for pred, ref, terms in zip(
        predictions,
        references,
        medical_terms_list
    ):

        candidates = set(
            parse_medical_terms(terms)
        )

        ref_present = [
            term
            for term in candidates
            if phrase_present(ref, term)
        ]

        if not ref_present:
            continue

        pred_present = [
            term
            for term in candidates
            if phrase_present(pred, term)
        ]

        ref_med_sentences.append(
            " ".join(sorted(ref_present))
        )

        pred_med_sentences.append(
            " ".join(sorted(pred_present))
        )


    if not ref_med_sentences:
        return np.nan


    return wer(
        ref_med_sentences,
        pred_med_sentences
    )


# ============================================================
# PER-SAMPLE mWER
# ============================================================

def sample_medical_wer(
    prediction,
    reference,
    medical_terms
):

    candidates = set(
        parse_medical_terms(medical_terms)
    )


    ref_present = sorted([
        term
        for term in candidates
        if phrase_present(reference, term)
    ])


    if len(ref_present) == 0:
        return np.nan


    pred_present = sorted([
        term
        for term in candidates
        if phrase_present(prediction, term)
    ])


    reference_medical_string = " ".join(
        ref_present
    )

    prediction_medical_string = " ".join(
        pred_present
    )


    return wer(
        reference_medical_string,
        prediction_medical_string
    )

In [8]:
# ============================================================
# SAMPLE ID -- SAME LOGIC AS YOUR WHISPER NOTEBOOK
# ============================================================

def make_sample_id(row):

    candidate_columns = [
        "sample_id",
        "utterance_id",
        "id",
        "file_id"
    ]

    for col in candidate_columns:

        if (
            col in row.index
            and pd.notna(row[col])
            and str(row[col]).strip()
        ):
            return str(row[col]).strip()


    audio_name = Path(
        str(row["audio_path"])
    ).name

    key = (
        f"{audio_name}|"
        f"{normalize_text(row['pidgin_transcription'])}"
    )

    return hashlib.sha1(
        key.encode("utf-8")
    ).hexdigest()[:20]

In [9]:
def prepare_shared_test_set():

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    df = pd.read_csv(
        METADATA_PATH
    )

    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
    )


    # --------------------------------------------------------
    # Support either spelling
    # --------------------------------------------------------

    medical_candidates = [
        "medical_terms_pidgin",
        "medical_terms_pidgen",
        "medical_terms_clean",
        "medical_terms",
    ]

    medical_col = next(
        (
            col
            for col in medical_candidates
            if col in df.columns
        ),
        None
    )

    if medical_col is None:
        raise ValueError(
            "Could not find a medical-term column."
        )


    print(
        "Using medical-term column:",
        medical_col
    )


    # --------------------------------------------------------
    # Standardized medical_terms
    # --------------------------------------------------------

    df["medical_terms"] = (
        df[medical_col]
        .apply(parse_medical_terms)
    )


    # --------------------------------------------------------
    # Remove [] / empty medical-term rows
    # --------------------------------------------------------

    before = len(df)

    df = df[
        df["medical_terms"].apply(
            lambda x: isinstance(x, list)
            and len(x) > 0
        )
    ].copy()


    print(
        f"Removed empty medical-term rows: "
        f"{before - len(df)}"
    )


    # --------------------------------------------------------
    # Reference
    # --------------------------------------------------------

    df["reference"] = (
        df["pidgin_transcription"]
        .apply(normalize_text)
    )


    # --------------------------------------------------------
    # Sample ID
    # --------------------------------------------------------

    if "sample_id" not in df.columns:

        df["sample_id"] = df.apply(
            make_sample_id,
            axis=1
        )

    else:

        missing_id = (
            df["sample_id"].isna()
            |
            (
                df["sample_id"]
                .astype(str)
                .str.strip()
                == ""
            )
        )

        if missing_id.any():

            df.loc[
                missing_id,
                "sample_id"
            ] = df.loc[
                missing_id
            ].apply(
                make_sample_id,
                axis=1
            )


    df["sample_id"] = (
        df["sample_id"]
        .astype(str)
    )


    # --------------------------------------------------------
    # Load shared split
    # --------------------------------------------------------

    manifest = pd.read_csv(
        SPLIT_MANIFEST_PATH,
        dtype={
            "sample_id": str
        }
    )


    if "split" not in manifest.columns:
        raise ValueError(
            "shared_split_manifest.csv "
            "does not contain 'split'."
        )


    manifest["sample_id"] = (
        manifest["sample_id"]
        .astype(str)
    )


    split_lookup = (
        manifest[
            ["sample_id", "split"]
        ]
        .drop_duplicates(
            subset="sample_id"
        )
    )


    # Remove existing split to avoid conflicts
    df = df.drop(
        columns=["split"],
        errors="ignore"
    )


    df = df.merge(
        split_lookup,
        on="sample_id",
        how="inner",
        validate="one_to_one"
    )


    # --------------------------------------------------------
    # TEST ONLY
    # --------------------------------------------------------

    df = df[
        df["split"]
        .astype(str)
        .str.lower()
        .eq("test")
    ].copy()


    # --------------------------------------------------------
    # Stronger medical check:
    #
    # Medical terms should not merely exist in the Excel cell;
    # at least one must actually occur in the reference.
    #
    # This guarantees per-row mWER is defined.
    # --------------------------------------------------------

    df["_reference_medical_terms"] = (
        df.apply(
            lambda row:
                reference_medical_terms(
                    row["reference"],
                    row["medical_terms"]
                ),
            axis=1
        )
    )


    before_reference_filter = len(df)


    df = df[
        df[
            "_reference_medical_terms"
        ].apply(len) > 0
    ].copy()


    print(
        "Removed rows where the medical-term list "
        "contained no term actually found in the reference:",
        before_reference_filter - len(df)
    )


    # --------------------------------------------------------
    # Resolve audio paths
    # --------------------------------------------------------

    def resolve_audio_path(path_value):

        path = Path(
            str(path_value)
        )

        if path.is_absolute():
            return str(path)

        return str(
            BASE_DIR / path
        )


    df["audio_filepath"] = (
        df["audio_path"]
        .apply(resolve_audio_path)
    )


    # --------------------------------------------------------
    # Verify audio
    # --------------------------------------------------------

    exists = (
        df["audio_filepath"]
        .apply(lambda x: Path(x).exists())
    )


    if not exists.all():

        missing = df.loc[
            ~exists,
            [
                "audio_path",
                "audio_filepath"
            ]
        ]

        print(
            "WARNING: missing audio:",
            len(missing)
        )

        display(
            missing.head(10)
        )

        df = df[
            exists
        ].copy()


    df = (
        df
        .sort_values("sample_id")
        .reset_index(drop=True)
    )


    print()
    print(
        "FINAL SHARED TEST SAMPLES:",
        len(df)
    )

    print(
        "Unique sample IDs:",
        df["sample_id"].nunique()
    )

    print(
        "All medical lists non-empty:",
        df["medical_terms"]
        .apply(len)
        .gt(0)
        .all()
    )

    print(
        "All samples contain ≥1 reference medical term:",
        df["_reference_medical_terms"]
        .apply(len)
        .gt(0)
        .all()
    )


    return df


test_df = prepare_shared_test_set()

display(
    test_df[
        [
            "sample_id",
            "audio_path",
            "reference",
            "medical_terms"
        ]
    ].head()
)

Using medical-term column: medical_terms_pidgen
Removed empty medical-term rows: 283
Removed rows where the medical-term list contained no term actually found in the reference: 6

FINAL SHARED TEST SAMPLES: 2217
Unique sample IDs: 2217
All medical lists non-empty: True
All samples contain ≥1 reference medical term: True


,sample_id,audio_path,reference,medical_terms
0,0007eceea7445996cb66,root/audio/Speaker_17/Speaker17-129.wav,hello e fit be some age related cortical chang...,"[cortical changes, computed tomography, normal..."
1,001275179d737b980aab,root/audio/Speaker_6/Speaker6-372.wav,after dem pull all my remaining teeth for lowe...,"[lower jaw, lump, cyst, hospital, lump reduce,..."
2,00197304cdccf249be4e,root/audio/Speaker_6/Speaker6-415.wav,my son na five months old and he don dey make ...,"[son, five months old, catarrh phlegm full, ch..."
3,0023b3f0f2acb32f7772,root/audio/Speaker_6/Speaker6-174.wav,go see lung doctor pulmonologist make dem do c...,"[lung, chest x ray spirometry breathing test, ..."
4,002922c6c57a34d7f082,root/audio/Speaker_1/Speaker1-157.wav,a couple of days ago my pulse rate dey fifty e...,"[a couple of days ago, pulse rate dey fifty ei..."


In [10]:
def add_evaluation_metrics(
    frame,
    model_name
):

    frame = frame.copy()


    # --------------------------------------------------------
    # Normalize reference and prediction consistently
    # --------------------------------------------------------

    frame["reference"] = (
        frame["reference"]
        .apply(normalize_text)
    )

    frame["prediction"] = (
        frame["prediction"]
        .apply(normalize_text)
    )


    # --------------------------------------------------------
    # WER
    # --------------------------------------------------------

    frame["wer"] = frame.apply(
        lambda row:
            wer(
                row["reference"],
                row["prediction"]
            ),
        axis=1
    )


    # --------------------------------------------------------
    # CER
    # --------------------------------------------------------

    frame["cer"] = frame.apply(
        lambda row:
            cer(
                row["reference"],
                row["prediction"]
            ),
        axis=1
    )


    # --------------------------------------------------------
    # mWER
    # --------------------------------------------------------

    frame["mwer"] = frame.apply(
        lambda row:
            sample_medical_wer(
                prediction=row["prediction"],
                reference=row["reference"],
                medical_terms=row["medical_terms"],
            ),
        axis=1
    )


    # --------------------------------------------------------
    # Useful downstream medical information
    # --------------------------------------------------------

    frame["reference_medical_terms"] = (
        frame.apply(
            lambda row:
                sorted(
                    reference_medical_terms(
                        row["reference"],
                        row["medical_terms"]
                    )
                ),
            axis=1
        )
    )


    frame["predicted_medical_terms"] = (
        frame.apply(
            lambda row:
                sorted(
                    predicted_medical_terms(
                        row["prediction"],
                        row["medical_terms"]
                    )
                ),
            axis=1
        )
    )


    frame["medical_exact_correct"] = (
        frame.apply(
            lambda row:
                int(
                    set(
                        row["reference_medical_terms"]
                    )
                    ==
                    set(
                        row["predicted_medical_terms"]
                    )
                ),
            axis=1
        )
    )


    frame["medical_num_reference"] = (
        frame[
            "reference_medical_terms"
        ].apply(len)
    )


    frame["medical_num_recovered"] = (
        frame.apply(
            lambda row:
                len(
                    set(
                        row[
                            "reference_medical_terms"
                        ]
                    )
                    &
                    set(
                        row[
                            "predicted_medical_terms"
                        ]
                    )
                ),
            axis=1
        )
    )


    frame["medical_recall"] = (

        frame[
            "medical_num_recovered"
        ]

        /

        frame[
            "medical_num_reference"
        ]
    )


    frame["model"] = model_name


    return frame

In [11]:
FEATURE_COLUMNS = [
    "audio_length",
    "speech_rate",
    "signal_to_noise_ratio",
    "silence_ratio",
    "english_ratio",
    "pidgin_ratio",
    "cmi",
    "code_switch_category",
]


def standardized_output(
    evaluated_df
):

    base_columns = [
        "sample_id",
        "audio_path",
        "audio_filepath",
        "speaker_id",
        "pidgin_transcription",
        "reference",
        "prediction",
        "medical_terms",
        "reference_medical_terms",
        "predicted_medical_terms",
        "medical_exact_correct",
        "medical_num_reference",
        "medical_num_recovered",
        "medical_recall",
        "wer",
        "cer",
        "mwer",
        "model",
    ]


    columns = [
        col
        for col in (
            base_columns
            + FEATURE_COLUMNS
        )
        if col in evaluated_df.columns
    ]


    result = evaluated_df[
        columns
    ].copy()


    # StatisticalSignificanceAnalysis.ipynb expects this
    # terminology in several places.
    result["medical_terms_pidgen"] = (
        result["medical_terms"]
    )


    return result

In [12]:
# ============================================================
# LOAD FINE-TUNED WHISPER-LARGE-V3
# ============================================================

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
)
import torch


print("Loading Whisper from:")
print(WHISPER_ROOT)


# ------------------------------------------------------------
# Processor
# ------------------------------------------------------------

whisper_processor = WhisperProcessor.from_pretrained(
    str(WHISPER_ROOT)
)


# ------------------------------------------------------------
# Model dtype
# ------------------------------------------------------------

whisper_dtype = (
    torch.float16
    if DEVICE == "cuda"
    else torch.float32
)


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

whisper_model = WhisperForConditionalGeneration.from_pretrained(
    str(WHISPER_ROOT),
    torch_dtype=whisper_dtype,
)


whisper_model = whisper_model.to(DEVICE)
whisper_model.eval()


# ------------------------------------------------------------
# Generation configuration
# ------------------------------------------------------------

whisper_model.generation_config.language = "en"
whisper_model.generation_config.task = "transcribe"


print("\nWhisper loaded successfully.")
print("Device:", next(whisper_model.parameters()).device)
print("dtype:", next(whisper_model.parameters()).dtype)
print("Model:", whisper_model.config._name_or_path)

Loading Whisper from:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginMed-ASR/whisper/hyperparameter_search/trial_01/selected_model


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]


Whisper loaded successfully.
Device: cuda:0
dtype: torch.float16
Model: /content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginMed-ASR/whisper/hyperparameter_search/trial_01/selected_model


In [13]:
# ============================================================
# LOAD FINE-TUNED GOOGLE MEDASR
# ============================================================

from transformers import (
    AutoProcessor,
    AutoModelForCTC,
)


print("Loading MedASR from:")
print(MEDASR_ROOT)


medasr_processor = (
    AutoProcessor.from_pretrained(
        MEDASR_ROOT
    )
)


medasr_model = (
    AutoModelForCTC.from_pretrained(
        MEDASR_ROOT
    )
)


medasr_model.to(DEVICE)
medasr_model.eval()


print()
print("MedASR loaded successfully.")
print("Device:", DEVICE)
print(
    "Parameters:",
    f"{sum(p.numel() for p in medasr_model.parameters()):,}"
)

Loading MedASR from:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/medasr_hf_clean/selected_model


Loading weights:   0%|          | 0/368 [00:00<?, ?it/s]


MedASR loaded successfully.
Device: cuda
Parameters: 105,265,408


In [14]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading Canary from:")
print(CANARY_ROOT)

canary_model = EncDecMultiTaskModel.restore_from(
    restore_path=str(CANARY_ROOT),
    map_location=torch.device(DEVICE),
)

canary_model = canary_model.to(DEVICE)
canary_model.eval()

print("\n✓ Canary loaded successfully")
print("Device:", next(canary_model.parameters()).device)
print(
    "Parameters:",
    f"{sum(p.numel() for p in canary_model.parameters()):,}"
)

Loading Canary from:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/selected_best_model.nemo
[NeMo I 2026-10-12 11:35:48 mixins:218] _setup_tokenizer: detected an aggregate tokenizer
[NeMo I 2026-10-12 11:35:48 mixins:357] Tokenizer SentencePieceTokenizer initialized with 1152 tokens
[NeMo I 2026-10-12 11:35:48 mixins:357] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-12 11:35:48 mixins:357] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-12 11:35:48 mixins:357] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-12 11:35:48 mixins:357] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-12 11:35:48 aggregate_tokenizer:73] Aggregate vocab size: 5248


[NeMo W 2026-10-12 11:35:48 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    tarred_audio_filepaths: null
    manifest_filepath: /content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/manifests/train_canary.jsonl
    sample_rate: 16000
    shuffle: true
    num_workers: 8
    batch_size: 16
    batch_duration: 360
    quadratic_duration: 15
    use_bucketing: true
    num_buckets: 20
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    text_field: text
    lang_field: target_lang
    
[NeMo W 2026-10-12 11:35:48 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation c

[NeMo I 2026-10-12 11:35:51 save_restore_connector:287] Model EncDecMultiTaskModel was successfully restored from /content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/selected_best_model.nemo.

✓ Canary loaded successfully
Device: cuda:0
Parameters: 182,643,840


In [15]:
# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Output directory:")
print(OUTPUT_DIR)


# ============================================================
# COMMON AUDIO LOADER
# ============================================================

import librosa
import soundfile as sf


def load_audio_16k(audio_path):

    audio, sr = sf.read(
        str(audio_path),
        dtype="float32"
    )

    # Stereo / multi-channel -> mono
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    # Resample if needed
    if sr != 16000:

        audio = librosa.resample(
            audio,
            orig_sr=sr,
            target_sr=16000
        )

        sr = 16000

    return audio.astype(np.float32), sr

Output directory:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/output


In [16]:
# ============================================================
# EVALUATE FINE-TUNED WHISPER-LARGE-V3
# ============================================================

whisper_predictions = []


for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Whisper-large-v3"
):

    audio, sr = load_audio_16k(
        row["audio_filepath"]
    )


    inputs = whisper_processor(
        audio,
        sampling_rate=sr,
        return_tensors="pt"
    )


    input_features = (
        inputs.input_features
        .to(
            device=DEVICE,
            dtype=whisper_model.dtype
        )
    )


    with torch.inference_mode():

        generated_ids = whisper_model.generate(
            input_features,
            language="en",
            task="transcribe",
            max_new_tokens=256
        )


    prediction = (
        whisper_processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )[0]
    )


    whisper_predictions.append(
        normalize_text(prediction)
    )


# ------------------------------------------------------------
# Attach predictions
# ------------------------------------------------------------

whisper_eval_df = test_df.copy()

whisper_eval_df[
    "prediction"
] = whisper_predictions


# ------------------------------------------------------------
# WER / CER / mWER / medical metrics
# ------------------------------------------------------------

whisper_eval_df = add_evaluation_metrics(
    whisper_eval_df,
    model_name="Whisper-large-v3"
)


# ------------------------------------------------------------
# Standardized output
# ------------------------------------------------------------

whisper_output_df = standardized_output(
    whisper_eval_df
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

WHISPER_OUTPUT_PATH = (
    OUTPUT_DIR
    / "Whisper_test_predictions.csv"
)


whisper_output_df.to_csv(
    WHISPER_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)


print("\nSaved:")
print(WHISPER_OUTPUT_PATH)

print(
    "Rows:",
    len(whisper_output_df)
)

display(
    whisper_output_df[
        [
            "sample_id",
            "reference",
            "prediction",
            "wer",
            "cer",
            "mwer",
            "medical_recall"
        ]
    ].head()
)

Whisper-large-v3:   0%|          | 0/2217 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_ne


Saved:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/output/Whisper_test_predictions.csv
Rows: 2217


,sample_id,reference,prediction,wer,cer,mwer,medical_recall
0,0007eceea7445996cb66,hello e fit be some age related cortical chang...,how far e fit be some age related critical cha...,0.130435,0.058577,0.500000,0.500000
1,001275179d737b980aab,after dem pull all my remaining teeth for lowe...,after dem pull all my remaining teeth for lowe...,0.043478,0.020528,0.000000,1.000000
2,00197304cdccf249be4e,my son na five months old and he don dey make ...,my son na five months old and he don dey make ...,0.225806,0.151376,0.333333,0.727273
3,0023b3f0f2acb32f7772,go see lung doctor pulmonologist make dem do c...,go see lung doctor make dem do chest x ray spi...,0.107143,0.189542,0.461538,0.750000
4,002922c6c57a34d7f082,a couple of days ago my pulse rate dey fifty e...,a couple of days ago my pulse rate dey now wey...,0.140351,0.120996,0.363636,0.700000


In [17]:
# ============================================================
# WHISPER CORPUS-LEVEL RESULTS
# ============================================================

whisper_overall = {
    "model": "Whisper-large-v3",

    "n_samples":
        len(whisper_output_df),

    "wer":
        wer(
            whisper_output_df[
                "reference"
            ].tolist(),

            whisper_output_df[
                "prediction"
            ].tolist()
        ),

    "cer":
        cer(
            whisper_output_df[
                "reference"
            ].tolist(),

            whisper_output_df[
                "prediction"
            ].tolist()
        ),

    "mwer":
        medical_wer(
            predictions=
                whisper_output_df[
                    "prediction"
                ].tolist(),

            references=
                whisper_output_df[
                    "reference"
                ].tolist(),

            medical_terms_list=
                whisper_output_df[
                    "medical_terms"
                ].tolist()
        ),

    "medical_recall":
        (
            whisper_output_df[
                "medical_num_recovered"
            ].sum()
            /
            whisper_output_df[
                "medical_num_reference"
            ].sum()
        ),

    "medical_exact_match":
        whisper_output_df[
            "medical_exact_correct"
        ].mean(),

    # useful for statistical analyses,
    # but not the main corpus WER
    "mean_sample_wer":
        whisper_output_df["wer"].mean(),

    "mean_sample_cer":
        whisper_output_df["cer"].mean(),

    "mean_sample_mwer":
        whisper_output_df["mwer"].mean()
}


whisper_overall

{'model': 'Whisper-large-v3',
 'n_samples': 2217,
 'wer': 0.2230610040218141,
 'cer': 0.1766786987253066,
 'mwer': 0.269766677620769,
 'medical_recall': np.float64(0.7745456656796145),
 'medical_exact_match': np.float64(0.33062697338746055),
 'mean_sample_wer': np.float64(0.17908032434964674),
 'mean_sample_cer': np.float64(0.13304953953585805),
 'mean_sample_mwer': np.float64(0.23051230055928648)}

In [19]:
# ============================================================
# EVALUATE FINE-TUNED GOOGLE MEDASR
# ============================================================

MEDASR_BATCH_SIZE = 16

medasr_predictions = []


for start in tqdm(
    range(
        0,
        len(test_df),
        MEDASR_BATCH_SIZE
    ),
    desc="Google MedASR"
):

    batch_df = test_df.iloc[
        start:
        start + MEDASR_BATCH_SIZE
    ]


    # --------------------------------------------------------
    # Load audio batch
    # --------------------------------------------------------

    audio_batch = []

    for audio_path in batch_df["audio_filepath"]:

        audio, sr = load_audio_16k(
            audio_path
        )

        audio_batch.append(
            audio
        )


    # --------------------------------------------------------
    # Processor
    # --------------------------------------------------------

    inputs = medasr_processor(
        audio_batch,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )


    # --------------------------------------------------------
    # Determine correct model input automatically
    # --------------------------------------------------------

    model_inputs = {}

    if "input_features" in inputs:

        model_inputs["input_features"] = (
            inputs["input_features"]
            .to(DEVICE)
        )

    elif "input_values" in inputs:

        model_inputs["input_values"] = (
            inputs["input_values"]
            .to(DEVICE)
        )

    else:

        raise KeyError(
            "Processor returned neither "
            "'input_features' nor 'input_values'. "
            f"Available keys: {list(inputs.keys())}"
        )


    # --------------------------------------------------------
    # Optional attention mask
    # --------------------------------------------------------

    if (
        "attention_mask" in inputs
        and
        inputs["attention_mask"] is not None
    ):

        model_inputs["attention_mask"] = (
            inputs["attention_mask"]
            .to(DEVICE)
        )


    # --------------------------------------------------------
    # Inference
    # --------------------------------------------------------

    with torch.inference_mode():

        outputs = medasr_model(
            **model_inputs
        )

        logits = outputs.logits


    # --------------------------------------------------------
    # CTC greedy decoding
    # --------------------------------------------------------

    predicted_ids = torch.argmax(
        logits,
        dim=-1
    )


    batch_predictions = (
        medasr_processor.batch_decode(
            predicted_ids
        )
    )


    medasr_predictions.extend([
        normalize_text(x)
        for x in batch_predictions
    ])


# ============================================================
# VALIDATION
# ============================================================

assert (
    len(medasr_predictions)
    ==
    len(test_df)
), (
    f"Prediction count mismatch: "
    f"{len(medasr_predictions)} "
    f"vs {len(test_df)}"
)


print(
    "Predictions generated:",
    len(medasr_predictions)
)


# ============================================================
# BUILD EVALUATION DATAFRAME
# ============================================================

medasr_eval_df = test_df.copy()

medasr_eval_df[
    "prediction"
] = medasr_predictions


# ============================================================
# WER / CER / mWER / MEDICAL METRICS
# ============================================================

medasr_eval_df = add_evaluation_metrics(
    medasr_eval_df,
    model_name="Google-MedASR"
)


# ============================================================
# STANDARDIZE OUTPUT
# ============================================================

medasr_output_df = standardized_output(
    medasr_eval_df
)


# ============================================================
# SAVE
# ============================================================

MEDASR_OUTPUT_PATH = (
    OUTPUT_DIR
    / "MedASR_test_predictions.csv"
)


medasr_output_df.to_csv(
    MEDASR_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)


print("\nSaved:")
print(MEDASR_OUTPUT_PATH)

print(
    "Rows:",
    len(medasr_output_df)
)


display(
    medasr_output_df[
        [
            "sample_id",
            "reference",
            "prediction",
            "wer",
            "cer",
            "mwer",
            "medical_recall"
        ]
    ].head()
)

Google MedASR:   0%|          | 0/139 [00:00<?, ?it/s]

Predictions generated: 2217

Saved:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/output/MedASR_test_predictions.csv
Rows: 2217


,sample_id,reference,prediction,wer,cer,mwer,medical_recall
0,0007eceea7445996cb66,hello e fit be some age related cortical chang...,howver e fit be some age related cortical chan...,0.282609,0.108787,0.500000,0.500000
1,001275179d737b980aab,after dem pull all my remaining teeth for lowe...,after de impve all my remaining teeth for lowe...,0.362319,0.187683,0.500000,0.666667
2,00197304cdccf249be4e,my son na five months old and he don dey make ...,my son na five months old and he don dey make ...,0.268817,0.160550,0.200000,0.909091
3,0023b3f0f2acb32f7772,go see lung doctor pulmonologist make dem do c...,go see lung doctor make dem do check x x ray s...,0.214286,0.228758,0.461538,0.750000
4,002922c6c57a34d7f082,a couple of days ago my pulse rate dey fifty e...,a couple of days ago my pulse rate day fift ei...,0.298246,0.128114,0.409091,0.600000


In [20]:
# ============================================================
# MEDASR CORPUS-LEVEL RESULTS
# ============================================================

medasr_overall = {
    "model": "Google-MedASR",

    "n_samples":
        len(medasr_output_df),

    "wer":
        wer(
            medasr_output_df[
                "reference"
            ].tolist(),

            medasr_output_df[
                "prediction"
            ].tolist()
        ),

    "cer":
        cer(
            medasr_output_df[
                "reference"
            ].tolist(),

            medasr_output_df[
                "prediction"
            ].tolist()
        ),

    "mwer":
        medical_wer(
            predictions=
                medasr_output_df[
                    "prediction"
                ].tolist(),

            references=
                medasr_output_df[
                    "reference"
                ].tolist(),

            medical_terms_list=
                medasr_output_df[
                    "medical_terms"
                ].tolist()
        ),

    "medical_recall":
        (
            medasr_output_df[
                "medical_num_recovered"
            ].sum()
            /
            medasr_output_df[
                "medical_num_reference"
            ].sum()
        ),

    "medical_exact_match":
        medasr_output_df[
            "medical_exact_correct"
        ].mean(),

    "mean_sample_wer":
        medasr_output_df[
            "wer"
        ].mean(),

    "mean_sample_cer":
        medasr_output_df[
            "cer"
        ].mean(),

    "mean_sample_mwer":
        medasr_output_df[
            "mwer"
        ].mean()
}


medasr_overall

{'model': 'Google-MedASR',
 'n_samples': 2217,
 'wer': 0.3030222175944606,
 'cer': 0.14851789202360097,
 'mwer': 0.41758133420966154,
 'medical_recall': np.float64(0.6484932938512454),
 'medical_exact_match': np.float64(0.11772665764546685),
 'mean_sample_wer': np.float64(0.30712835334592553),
 'mean_sample_cer': np.float64(0.14984927253618444),
 'mean_sample_mwer': np.float64(0.4044065066848947)}

In [21]:
# ============================================================
# CANARY OUTPUT NORMALIZER
# ============================================================

def extract_canary_text(output):

    if isinstance(
        output,
        str
    ):
        return output


    if hasattr(
        output,
        "text"
    ):
        return output.text


    if isinstance(
        output,
        dict
    ):

        if "text" in output:
            return output["text"]


    return str(output)

In [22]:
# ============================================================
# EVALUATE FINE-TUNED CANARY-180M
# ============================================================

CANARY_BATCH_SIZE = 16


audio_paths = (
    test_df[
        "audio_filepath"
    ]
    .astype(str)
    .tolist()
)


canary_predictions = []


for start in tqdm(
    range(
        0,
        len(audio_paths),
        CANARY_BATCH_SIZE
    ),
    desc="Canary-180M"
):

    batch_paths = audio_paths[
        start:
        start + CANARY_BATCH_SIZE
    ]


    with torch.inference_mode():

        outputs = canary_model.transcribe(
            batch_paths,
            batch_size=len(
                batch_paths
            )
        )


    for output in outputs:

        prediction = extract_canary_text(
            output
        )

        canary_predictions.append(
            normalize_text(
                prediction
            )
        )


assert (
    len(canary_predictions)
    ==
    len(test_df)
), (
    f"Canary prediction count mismatch: "
    f"{len(canary_predictions)} "
    f"vs {len(test_df)}"
)


# ------------------------------------------------------------
# Evaluation DataFrame
# ------------------------------------------------------------

canary_eval_df = test_df.copy()

canary_eval_df[
    "prediction"
] = canary_predictions


canary_eval_df = add_evaluation_metrics(
    canary_eval_df,
    model_name="Canary-180M"
)


canary_output_df = standardized_output(
    canary_eval_df
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

CANARY_OUTPUT_PATH = (
    OUTPUT_DIR
    / "Canary_test_predictions.csv"
)


canary_output_df.to_csv(
    CANARY_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)


print("\nSaved:")
print(CANARY_OUTPUT_PATH)

print(
    "Rows:",
    len(canary_output_df)
)

Canary-180M:   0%|          | 0/139 [00:00<?, ?it/s]

[NeMo W 2026-10-12 13:16:23 aed_multitask_models:607] Chunking is disabled. Please pass a single audio file or set batch_size to 1
[NeMo W 2026-10-12 13:16:23 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: trim_silence,enable_chunking
[NeMo W 2026-10-12 13:16:23 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:04,  4.61s/it]
[NeMo W 2026-10-12 13:16:28 aed_multitask_models:607] Chunking is disabled. Please pass a single audio file or set batch_size to 1
[NeMo W 2026-10-12 13:16:28 dataloader:881] The following configuration keys are ignored by


Saved:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/output/Canary_test_predictions.csv
Rows: 2217


In [23]:
# ============================================================
# CANARY CORPUS-LEVEL RESULTS
# ============================================================

canary_overall = {
    "model": "Canary-180M",

    "n_samples":
        len(canary_output_df),

    "wer":
        wer(
            canary_output_df[
                "reference"
            ].tolist(),

            canary_output_df[
                "prediction"
            ].tolist()
        ),

    "cer":
        cer(
            canary_output_df[
                "reference"
            ].tolist(),

            canary_output_df[
                "prediction"
            ].tolist()
        ),

    "mwer":
        medical_wer(
            predictions=
                canary_output_df[
                    "prediction"
                ].tolist(),

            references=
                canary_output_df[
                    "reference"
                ].tolist(),

            medical_terms_list=
                canary_output_df[
                    "medical_terms"
                ].tolist()
        ),

    "medical_recall":
        (
            canary_output_df[
                "medical_num_recovered"
            ].sum()
            /
            canary_output_df[
                "medical_num_reference"
            ].sum()
        ),

    "medical_exact_match":
        canary_output_df[
            "medical_exact_correct"
        ].mean(),

    "mean_sample_wer":
        canary_output_df[
            "wer"
        ].mean(),

    "mean_sample_cer":
        canary_output_df[
            "cer"
        ].mean(),

    "mean_sample_mwer":
        canary_output_df[
            "mwer"
        ].mean()
}


canary_overall

{'model': 'Canary-180M',
 'n_samples': 2217,
 'wer': 0.26863505616222616,
 'cer': 0.20968253488793573,
 'mwer': 0.40453499835688467,
 'medical_recall': np.float64(0.6468094989258549),
 'medical_exact_match': np.float64(0.15877311682453765),
 'mean_sample_wer': np.float64(0.27924330817448434),
 'mean_sample_cer': np.float64(0.22299452921150886),
 'mean_sample_mwer': np.float64(0.39352909896765503)}

In [24]:
# ============================================================
# VERIFY MODEL ALIGNMENT
# ============================================================

assert (
    len(whisper_output_df)
    ==
    len(canary_output_df)
    ==
    len(medasr_output_df)
), "The three models have different numbers of samples."


assert (
    whisper_output_df[
        "sample_id"
    ].tolist()
    ==
    canary_output_df[
        "sample_id"
    ].tolist()
    ==
    medasr_output_df[
        "sample_id"
    ].tolist()
), "sample_id ordering differs between models."


assert (
    whisper_output_df[
        "reference"
    ].tolist()
    ==
    canary_output_df[
        "reference"
    ].tolist()
    ==
    medasr_output_df[
        "reference"
    ].tolist()
), "References differ between model outputs."


assert whisper_output_df[
    "medical_terms"
].apply(
    lambda x: len(
        parse_medical_terms(x)
    ) > 0
).all()


assert canary_output_df[
    "medical_terms"
].apply(
    lambda x: len(
        parse_medical_terms(x)
    ) > 0
).all()


assert medasr_output_df[
    "medical_terms"
].apply(
    lambda x: len(
        parse_medical_terms(x)
    ) > 0
).all()


print(
    "✓ All models evaluated on exactly",
    len(whisper_output_df),
    "identical samples."
)

print(
    "✓ No empty medical-term rows."
)

print(
    "✓ References and sample ordering match."
)

✓ All models evaluated on exactly 2217 identical samples.
✓ No empty medical-term rows.
✓ References and sample ordering match.


In [25]:
# ============================================================
# OVERALL RESULTS
# ============================================================

overall_results_df = pd.DataFrame([
    whisper_overall,
    canary_overall,
    medasr_overall
])


OVERALL_RESULTS_PATH = (
    OUTPUT_DIR
    / "overall_model_metrics.csv"
)


overall_results_df.to_csv(
    OVERALL_RESULTS_PATH,
    index=False,
    encoding="utf-8-sig"
)


display(
    overall_results_df
)


print("\nSaved:")
print(OVERALL_RESULTS_PATH)

,model,n_samples,wer,cer,mwer,medical_recall,medical_exact_match,mean_sample_wer,mean_sample_cer,mean_sample_mwer
0,Whisper-large-v3,2217,0.223061,0.176679,0.269767,0.774546,0.330627,0.179080,0.133050,0.230512
1,Canary-180M,2217,0.268635,0.209683,0.404535,0.646809,0.158773,0.279243,0.222995,0.393529
2,Google-MedASR,2217,0.303022,0.148518,0.417581,0.648493,0.117727,0.307128,0.149849,0.404407



Saved:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/output/overall_model_metrics.csv


In [26]:
# ============================================================
# BUILD ALIGNED STATISTICAL ANALYSIS FILE
# ============================================================

base_columns = [
    "sample_id",
    "audio_path",
    "reference",
    "medical_terms",
]


whisper_stats = (
    whisper_output_df[
        base_columns
        + [
            "prediction",
            "wer",
            "cer",
            "mwer",
            "medical_recall",
            "medical_exact_correct"
        ]
    ]
    .rename(
        columns={
            "prediction":
                "whisper_prediction",

            "wer":
                "whisper_wer",

            "cer":
                "whisper_cer",

            "mwer":
                "whisper_mwer",

            "medical_recall":
                "whisper_medical_recall",

            "medical_exact_correct":
                "whisper_medical_exact"
        }
    )
)


canary_stats = (
    canary_output_df[
        [
            "sample_id",
            "prediction",
            "wer",
            "cer",
            "mwer",
            "medical_recall",
            "medical_exact_correct"
        ]
    ]
    .rename(
        columns={
            "prediction":
                "canary_prediction",

            "wer":
                "canary_wer",

            "cer":
                "canary_cer",

            "mwer":
                "canary_mwer",

            "medical_recall":
                "canary_medical_recall",

            "medical_exact_correct":
                "canary_medical_exact"
        }
    )
)


medasr_stats = (
    medasr_output_df[
        [
            "sample_id",
            "prediction",
            "wer",
            "cer",
            "mwer",
            "medical_recall",
            "medical_exact_correct"
        ]
    ]
    .rename(
        columns={
            "prediction":
                "medasr_prediction",

            "wer":
                "medasr_wer",

            "cer":
                "medasr_cer",

            "mwer":
                "medasr_mwer",

            "medical_recall":
                "medasr_medical_recall",

            "medical_exact_correct":
                "medasr_medical_exact"
        }
    )
)


aligned_stats_df = (
    whisper_stats
    .merge(
        canary_stats,
        on="sample_id",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        medasr_stats,
        on="sample_id",
        how="inner",
        validate="one_to_one"
    )
)


assert (
    len(aligned_stats_df)
    ==
    len(test_df)
)


ALIGNED_STATS_PATH = (
    OUTPUT_DIR
    / "aligned_model_predictions_for_statistics.csv"
)


aligned_stats_df.to_csv(
    ALIGNED_STATS_PATH,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Aligned statistical samples:",
    len(aligned_stats_df)
)

print(
    "Saved:",
    ALIGNED_STATS_PATH
)

display(
    aligned_stats_df.head()
)

Aligned statistical samples: 2217
Saved: /content/drive/MyDrive/Brock_University/pidgindataset2026October/output/aligned_model_predictions_for_statistics.csv


,sample_id,audio_path,reference,medical_terms,whisper_prediction,whisper_wer,whisper_cer,whisper_mwer,whisper_medical_recall,whisper_medical_exact,...,canary_cer,canary_mwer,canary_medical_recall,canary_medical_exact,medasr_prediction,medasr_wer,medasr_cer,medasr_mwer,medasr_medical_recall,medasr_medical_exact
0,0007eceea7445996cb66,root/audio/Speaker_17/Speaker17-129.wav,hello e fit be some age related cortical chang...,"[cortical changes, computed tomography, normal...",how far e fit be some age related critical cha...,0.130435,0.058577,0.500000,0.500000,0,...,0.100418,0.833333,0.250000,0,howver e fit be some age related cortical chan...,0.282609,0.108787,0.500000,0.500000,0
1,001275179d737b980aab,root/audio/Speaker_6/Speaker6-372.wav,after dem pull all my remaining teeth for lowe...,"[lower jaw, lump, cyst, hospital, lump reduce,...",after dem pull all my remaining teeth for lowe...,0.043478,0.020528,0.000000,1.000000,1,...,1.000000,1.000000,0.000000,0,after de impve all my remaining teeth for lowe...,0.362319,0.187683,0.500000,0.666667,0
2,00197304cdccf249be4e,root/audio/Speaker_6/Speaker6-415.wav,my son na five months old and he don dey make ...,"[son, five months old, catarrh phlegm full, ch...",my son na five months old and he don dey make ...,0.225806,0.151376,0.333333,0.727273,0,...,0.098624,0.200000,0.909091,0,my son na five months old and he don dey make ...,0.268817,0.160550,0.200000,0.909091,0
3,0023b3f0f2acb32f7772,root/audio/Speaker_6/Speaker6-174.wav,go see lung doctor pulmonologist make dem do c...,"[lung, chest x ray spirometry breathing test, ...",go see lung doctor make dem do chest x ray spi...,0.107143,0.189542,0.461538,0.750000,0,...,1.000000,1.000000,0.000000,0,go see lung doctor make dem do check x x ray s...,0.214286,0.228758,0.461538,0.750000,0
4,002922c6c57a34d7f082,root/audio/Speaker_1/Speaker1-157.wav,a couple of days ago my pulse rate dey fifty e...,"[a couple of days ago, pulse rate dey fifty ei...",a couple of days ago my pulse rate dey now wey...,0.140351,0.120996,0.363636,0.700000,0,...,0.060498,0.318182,0.700000,0,a couple of days ago my pulse rate day fift ei...,0.298246,0.128114,0.409091,0.600000,0


In [27]:
# ============================================================
# SAVE FINAL FILTERED TEST MANIFEST
# ============================================================

final_test_manifest = test_df[
    [
        "sample_id",
        "audio_path",
        "audio_filepath",
        "reference",
        "medical_terms",
        "split"
    ]
].copy()


TEST_MANIFEST_OUTPUT = (
    OUTPUT_DIR
    / "filtered_medical_test_manifest.csv"
)


final_test_manifest.to_csv(
    TEST_MANIFEST_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Saved:",
    TEST_MANIFEST_OUTPUT
)

Saved: /content/drive/MyDrive/Brock_University/pidgindataset2026October/output/filtered_medical_test_manifest.csv
